In [2]:
# Installation for mysql-connector-python
# !pip install mysql-connector-python

In [3]:
# Connect with mysql
import mysql.connector
connection = mysql.connector.connect(
    host="localhost",
    user=your_username,
    password=your_password,
    database="review_collection"
)

print("Connected successfully!")

Connected successfully!


In [4]:
# To check whether we are connecting to the right database
cursor = connection.cursor()
cursor.execute("SHOW TABLES")

for table in cursor:
    print(table)

('app_info',)
('ingestion_run',)
('processed_review',)
('raw_review',)
('review_ingestion',)
('review_quality',)


In [5]:
# app_info table

# app list
apps_dict = {
    "Snapchat": "com.snapchat.android", 
    "Discord": "com.discord",
    "Duolingo": "com.duolingo",
    "Early Learning Academy": "mobi.abcmouse.academy_goo",
    "YouTube": "com.google.android.youtube",
    "Prime Video": "com.amazon.avod.thirdpartyclient", 
    "WPS Office-PDF, Word, Sheet": "cn.wps.moffice_eng",
    "Claude by Anthropic": "com.anthropic.claude",
    "Spotify: Music and Podcasts": "com.spotify.music",
    "DuckDuckGo, optional Duck.ai": "com.duckduckgo.mobile.android"
}

sql_appinfo = "INSERT IGNORE INTO app_info (app_id, platform, app_name) VALUES (%s, %s, %s)"

for app_name, app_id in apps_dict.items():
    values_appinfo = (
        app_id,
        "Google Play",
        app_name
    )
    cursor.execute(sql_appinfo, values_appinfo)

connection.commit()

print("app_info inserted!")

app_info inserted!


In [6]:
# app_info table CHECK
cursor.execute("SELECT * FROM app_info")

for row in cursor.fetchall():
    print(row)

('cn.wps.moffice_eng', 'Google Play', 'WPS Office-PDF, Word, Sheet')
('com.amazon.avod.thirdpartyclient', 'Google Play', 'Prime Video')
('com.anthropic.claude', 'Google Play', 'Claude by Anthropic')
('com.discord', 'Google Play', 'Discord')
('com.duckduckgo.mobile.android', 'Google Play', 'DuckDuckGo, optional Duck.ai')
('com.duolingo', 'Google Play', 'Duolingo')
('com.google.android.youtube', 'Google Play', 'YouTube')
('com.snapchat.android', 'Google Play', 'Snapchat')
('com.spotify.music', 'Google Play', 'Spotify: Music and Podcasts')
('mobi.abcmouse.academy_goo', 'Google Play', 'Early Learning Academy')


In [34]:
from google_play_scraper import Sort, reviews
import pandas as pd
from datetime import datetime

# Settings
language = "en"
country = "us"
sort_method = "NEWEST"
target_count = 50

all_reviews=[]

# review collection
for app_name, app_id in apps_dict.items():
    result, continuation_token = reviews(
        app_id,
        sort=Sort.NEWEST,
        lang=language,
        country=country,
        count=target_count,
    )

    actual_count = len(result)
    print(app_name, actual_count)

# ingestion_run table
    sql = "INSERT INTO ingestion_run(app_id,platform,collect_at,language,country,sort_method,target_review_count,actual_review_count,skipped_duplicates,inserted_records,status,error_message) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)"
    values = (app_id, "Google Play", datetime.now(), language, country, sort_method, target_count, actual_count, 0, 0, "completed", None)

    cursor.execute(sql, values)
    run_id = cursor.lastrowid

    inserted_records = 0
    skipped_duplicates = 0
    
# raw_review table
    sql_raw = "INSERT IGNORE INTO raw_review(review_id,app_id,platform,user_name,content,rating,thumbs_up_count,review_time,developer_reply,developer_reply_time,app_version,review_created_version,ingested_at) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)"

    for review in result:
        values_raw = (
            review["reviewId"],
            app_id,
            "Google Play",
            review["userName"],
            review["content"],
            review["score"],
            review["thumbsUpCount"],
            review["at"],
            review["replyContent"],
            review["repliedAt"],
            review["appVersion"],
            review["reviewCreatedVersion"],
            datetime.now()
        )
        cursor.execute(sql_raw, values_raw)
        
        if cursor.rowcount == 1:
            inserted_records += 1
            raw_id = cursor.lastrowid

            #review_ingestion
            sql_ingestion = "INSERT INTO review_ingestion(raw_id,run_id,record_status) VALUES(%s,%s,%s)"
            values_ingestion = (
                raw_id,
                run_id,
                "Inserted"
            )
            cursor.execute(sql_ingestion, values_ingestion)
        else:
            skipped_duplicates += 1
            #review_ingestion
            sql_find = "SELECT raw_id FROM raw_review WHERE review_id=%s AND app_id=%s AND platform=%s"
            values_find = (
                review["reviewId"], 
                app_id, 
                "Google Play"
            )
            cursor.execute(sql_find,values_find)
            raw_id = cursor.fetchone()[0]
            sql_ingestion = "INSERT INTO review_ingestion(raw_id,run_id,record_status) VALUES(%s,%s,%s)"
            values_ingestion = (
                raw_id,
                run_id,
                "Duplicate"
            )
            cursor.execute(sql_ingestion,values_ingestion)
            
    # update ingestion_run
    sql_update = "UPDATE ingestion_run SET inserted_records = %s, skipped_duplicates = %s WHERE run_id = %s"

    cursor.execute(
        sql_update,
        (
            inserted_records,
            skipped_duplicates,
            run_id
        )
    )
    connection.commit()

    print(
        app_name,
        "finished:",
        inserted_records,
        "inserted,",
        skipped_duplicates,
        "duplicates"
    )
print("Pipeline completed!")

Snapchat 50
Snapchat finished: 0 inserted, 50 duplicates
Discord 50
Discord finished: 0 inserted, 50 duplicates
Duolingo 50
Duolingo finished: 0 inserted, 50 duplicates
Early Learning Academy 50
Early Learning Academy finished: 0 inserted, 50 duplicates
YouTube 50
YouTube finished: 29 inserted, 21 duplicates
Prime Video 50
Prime Video finished: 0 inserted, 50 duplicates
WPS Office-PDF, Word, Sheet 50
WPS Office-PDF, Word, Sheet finished: 0 inserted, 50 duplicates
Claude by Anthropic 50
Claude by Anthropic finished: 0 inserted, 50 duplicates
Spotify: Music and Podcasts 50
Spotify: Music and Podcasts finished: 0 inserted, 50 duplicates
DuckDuckGo, optional Duck.ai 50
DuckDuckGo, optional Duck.ai finished: 0 inserted, 50 duplicates
Pipeline completed!


In [35]:
# ingestion_run table CHECK
cursor.execute("SELECT * FROM ingestion_run")

for row in cursor.fetchall():
    print(row)

(1, 'com.snapchat.android', 'Google Play', datetime.datetime(2026, 7, 30, 1, 55, 29), 'en', 'us', 'NEWEST', 50, 50, 0, 50, 'completed', None)
(2, 'com.discord', 'Google Play', datetime.datetime(2026, 7, 30, 1, 55, 30), 'en', 'us', 'NEWEST', 50, 50, 0, 50, 'completed', None)
(3, 'com.duolingo', 'Google Play', datetime.datetime(2026, 7, 30, 1, 55, 32), 'en', 'us', 'NEWEST', 50, 50, 0, 50, 'completed', None)
(4, 'mobi.abcmouse.academy_goo', 'Google Play', datetime.datetime(2026, 7, 30, 1, 55, 33), 'en', 'us', 'NEWEST', 50, 50, 0, 50, 'completed', None)
(5, 'com.google.android.youtube', 'Google Play', datetime.datetime(2026, 7, 30, 1, 55, 34), 'en', 'us', 'NEWEST', 50, 50, 0, 50, 'completed', None)
(6, 'com.amazon.avod.thirdpartyclient', 'Google Play', datetime.datetime(2026, 7, 30, 1, 55, 36), 'en', 'us', 'NEWEST', 50, 50, 0, 50, 'completed', None)
(7, 'cn.wps.moffice_eng', 'Google Play', datetime.datetime(2026, 7, 30, 1, 55, 37), 'en', 'us', 'NEWEST', 50, 50, 0, 50, 'completed', None)
(8

In [36]:
# raw_review table CHECK
cursor.execute("SELECT * FROM raw_review LIMIT 10")

for row in cursor.fetchall():
    print(row)

(1, '023aed25-375a-4604-959d-b6e5c55e88aa', 'com.snapchat.android', 'Google Play', 'Ian Wakaba', "I've had a very disappointing experience with Snapchat. My main account of over five years became inaccessible, and when I created a new account, it was locked within a few days despite not posting any content. The only thing I did was add friends, many of whom I know. The app provides little explanation for why accounts are locked, and the appeal and recovery process is frustrating and unhelpful. Loyal users deserve clearer communication and better customer support instead of unexplained bans.", 1, 0, datetime.datetime(2026, 7, 29, 1, 54, 31), None, None, None, None, datetime.datetime(2026, 7, 30, 1, 55, 29))
(2, 'd95f7f61-e635-420b-a25a-84256849f001', 'com.snapchat.android', 'Google Play', 'Rigela Kapa', 'I love the feature with Snapchat! but add a privacy mode for the friends. they can see my location and some1 is after me', 3, 0, datetime.datetime(2026, 7, 29, 1, 52, 57), None, None, N

In [37]:
# review_ingestion table CHECK
cursor.execute("SELECT * FROM review_ingestion LIMIT 20")

for row in cursor.fetchall():
    print(row)

(1, 1, 'Inserted')
(1, 11, 'Duplicate')
(2, 1, 'Inserted')
(2, 11, 'Duplicate')
(3, 1, 'Inserted')
(3, 11, 'Duplicate')
(4, 1, 'Inserted')
(4, 11, 'Duplicate')
(5, 1, 'Inserted')
(5, 11, 'Duplicate')
(6, 1, 'Inserted')
(6, 11, 'Duplicate')
(7, 1, 'Inserted')
(7, 11, 'Duplicate')
(8, 1, 'Inserted')
(8, 11, 'Duplicate')
(9, 1, 'Inserted')
(9, 11, 'Duplicate')
(10, 1, 'Inserted')
(10, 11, 'Duplicate')


In [38]:
# processed_review table
import re

def remove_emoji(string):
    if string is None:
        return None
    
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002500-\U00002BEF"  # chinese char
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               u"\U0001f926-\U0001f937"
                               u"\U00010000-\U0010ffff"
                               u"\u2640-\u2642"
                               u"\u2600-\u2B55"
                               u"\u200d"
                               u"\u23cf"
                               u"\u23e9"
                               u"\u231a"
                               u"\ufe0f"  # dingbats
                               u"\u3030"
                               "]+", flags=re.UNICODE)
    
    return emoji_pattern.sub(r'', string)

def clean_review(text):

    if text is None:
        return None

    text = remove_emoji(text)
    text = text.lower()
    text = " ".join(text.split())
    text = text.strip()

    return text

sql_takeraw = "SELECT raw_id, content FROM raw_review"
cursor.execute(sql_takeraw)
raw_reviews = cursor.fetchall()

for raw_id, content in raw_reviews:

    cleaned_content = clean_review(content)
    if cleaned_content:
        content_length = len(cleaned_content)
    else:
        content_length = 0

    sql_process = "INSERT IGNORE INTO processed_review(raw_id,cleaned_content,content_length) VALUES (%s,%s,%s)"
    values_process = (
        raw_id,
        cleaned_content,
        content_length
    )
    cursor.execute(sql_process, values_process)

connection.commit()

print("processed_review completed!")

processed_review completed!


In [39]:
# raw_review preview
cursor.execute("SELECT raw_id, content FROM raw_review LIMIT 10")

for row in cursor.fetchall():
    print(row)

(1, "I've had a very disappointing experience with Snapchat. My main account of over five years became inaccessible, and when I created a new account, it was locked within a few days despite not posting any content. The only thing I did was add friends, many of whom I know. The app provides little explanation for why accounts are locked, and the appeal and recovery process is frustrating and unhelpful. Loyal users deserve clearer communication and better customer support instead of unexplained bans.")
(2, 'I love the feature with Snapchat! but add a privacy mode for the friends. they can see my location and some1 is after me')
(3, '👍🏻')
(4, 'good luck')
(5, 'Sandile Mtsh')
(6, 'AnkitKumar')
(7, 'Nice 👍🏻')
(8, 'very nice')
(9, 'fun!')
(10, 'I M Happy')


In [40]:
# processed_review table CHECK
cursor.execute("SELECT * FROM processed_review LIMIT 10")

for row in cursor.fetchall():
    print(row)

(1, "i've had a very disappointing experience with snapchat. my main account of over five years became inaccessible, and when i created a new account, it was locked within a few days despite not posting any content. the only thing i did was add friends, many of whom i know. the app provides little explanation for why accounts are locked, and the appeal and recovery process is frustrating and unhelpful. loyal users deserve clearer communication and better customer support instead of unexplained bans.", 499)
(2, 'i love the feature with snapchat! but add a privacy mode for the friends. they can see my location and some1 is after me', 120)
(3, '', 0)
(4, 'good luck', 9)
(5, 'sandile mtsh', 12)
(6, 'ankitkumar', 10)
(7, 'nice', 4)
(8, 'very nice', 9)
(9, 'fun!', 4)
(10, 'i m happy', 9)


In [41]:
# review_quality table
cursor.execute("SELECT raw_id,content,review_created_version,app_version,developer_reply,developer_reply_time FROM raw_review")
raw_reviews_more = cursor.fetchall()

seen_text = set()

for (raw_id,content,review_created_version,app_version,developer_reply,developer_reply_time) in raw_reviews_more:
    
    cleaned_content = clean_review(content)
    if cleaned_content:
        content_length = len(cleaned_content)
    else:
        content_length = 0

    # default values
    is_empty_content = 0
    is_repeated_text = 0
    is_low_signal = 0
    is_missing_created_version = 0
    is_missing_app_version = 0
    is_missing_developer_reply = 0
    is_missing_developer_reply_time = 0

    # empty content
    if cleaned_content is None or len(cleaned_content.strip()) == 0:
        is_empty_content = 1

    # repeated text
    if content:
        text = content.lower().strip()
        if text in seen_text:
            is_repeated_text = 1
        else:
            seen_text.add(text)
            
    # low signal
    if cleaned_content:
        if len(cleaned_content.strip()) < 5:
            is_low_signal = 1

    # missing fields
    if review_created_version is None:
        is_missing_created_version = 1
    if app_version is None:
        is_missing_app_version = 1
    if developer_reply is None:
        is_missing_developer_reply = 1
    if developer_reply_time is None:
        is_missing_developer_reply_time = 1

    # insert quality table
    sql_quality = "INSERT IGNORE INTO review_quality(raw_id,is_empty_content,is_repeated_text,is_low_signal,is_missing_created_version,is_missing_app_version,is_missing_developer_reply,is_missing_developer_reply_time) VALUES (%s,%s,%s,%s,%s,%s,%s,%s)"
    values_quality = (
        raw_id,
        is_empty_content,
        is_repeated_text,
        is_low_signal,
        is_missing_created_version,
        is_missing_app_version,
        is_missing_developer_reply,
        is_missing_developer_reply_time
    )
    cursor.execute(
        sql_quality,
        values_quality
    )

connection.commit()
print("review_quality completed!")

review_quality completed!


In [42]:
# review_quality table CHECK
cursor.execute("SELECT * FROM review_quality LIMIT 10")

for row in cursor.fetchall():
    print(row)

(1, 0, 0, 0, 1, 1, 1, 1)
(2, 0, 0, 0, 1, 1, 1, 1)
(3, 1, 0, 0, 0, 0, 1, 1)
(4, 0, 0, 0, 0, 0, 1, 1)
(5, 0, 0, 0, 1, 1, 1, 1)
(6, 0, 0, 0, 0, 0, 1, 1)
(7, 0, 0, 1, 0, 0, 1, 1)
(8, 0, 0, 0, 1, 1, 1, 1)
(9, 0, 0, 1, 0, 0, 1, 1)
(10, 0, 0, 0, 0, 0, 1, 1)
